# Datakwaliteit CHO-KENNIS

Dashboard op `data/historie.csv`, gevuld door `python kwaliteit.py`.

- **Endpoint:** `https://api.linkeddata.cultureelerfgoed.nl/datasets/rce/cho/sparql`
- **Gecontroleerd:** de hoofdgraph `graph/instanties-rce`
- **Naslag (niet zelf gecontroleerd):** OWMS, de waardelijst achter `ceo:heeftGemeente` en `ceo:heeftProvincie`. Die staat twee keer in de dataset (KOOP-origineel en `graph/owms`); check 23 bewaakt dat beide kopieën gelijk blijven.
- **Actieve rijksmonumenten** = `ceo:heeftJuridischeStatus` *rijksmonument* (niet: voorbeschermd of geen rijksmonument)

Zet `DRAAI_NU = True` om eerst een nieuwe run te doen.

In [ ]:
DRAAI_NU = False

import subprocess, sys
if DRAAI_NU:
    subprocess.run([sys.executable, "kwaliteit.py"], check=True)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# Statuskleuren: vast, en altijd samen met icoon + label (nooit kleur alleen)
STATUS = {
    "in orde":   ("#0ca30c", "✔"),
    "bevinding": ("#ec835a", "▲"),
    "mislukt":   ("#d03b3b", "✖"),
}
SERIE = "#2a78d6"
INKT, INKT_2, RASTER = "#0b0b0b", "#52514e", "#e4e3df"

h = pd.read_csv("data/historie.csv", dtype={"id": str})
h["run"] = pd.to_datetime(h["run"])
runs = sorted(h["run"].unique())
laatste = h[h["run"] == runs[-1]].copy()
vorige = h[h["run"] == runs[-2]].set_index("id") if len(runs) > 1 else None

def status(r):
    if r["status"] != "ok":
        return "mislukt"
    return "bevinding" if r["fout"] > 0 else "in orde"

laatste["oordeel"] = laatste.apply(status, axis=1)
if vorige is not None:
    laatste["verschil"] = laatste.apply(
        lambda r: r["fout"] - vorige["fout"].get(r["id"], float("nan")), axis=1)
else:
    laatste["verschil"] = float("nan")
print(f"Laatste run: {runs[-1]:%Y-%m-%d %H:%M} UTC  ·  {len(runs)} run(s) in historie")

## Overzicht

In [ ]:
tellingen = laatste["oordeel"].value_counts()
tegels = "".join(
    f'''<div style="flex:1;min-width:150px;border:1px solid {RASTER};border-radius:8px;padding:12px 16px">
          <div style="color:{INKT_2};font-size:13px">{STATUS[k][1]} {k}</div>
          <div style="color:{INKT};font-size:32px;font-weight:600">{tellingen.get(k, 0)}</div>
        </div>'''
    for k in STATUS)
display(HTML(f'<div style="display:flex;gap:12px;flex-wrap:wrap;font-family:sans-serif">{tegels}</div>'))

## Alle checks

In [ ]:
def label(o):
    kleur, icoon = STATUS[o]
    return f'<span style="white-space:nowrap"><span style="color:{kleur};font-weight:600">{icoon}</span> {o}</span>'

def getal(x, teken=False):
    if pd.isna(x):
        return ""
    s = f"{int(x):+,}" if teken else f"{int(x):,}"
    return s.replace(",", ".")

from pathlib import Path

def lijst_pad(cid):
    return Path("data/lijsten") / f"{cid}.csv"

def lijst_link(cid):
    p = lijst_pad(cid)
    if not p.exists():
        return ""
    n = sum(1 for _ in p.open(encoding="utf-8")) - 1
    return f'<a href="#lijst-{cid}">bekijk ({getal(n)})</a>'

tabel = pd.DataFrame({
    "Status": laatste["oordeel"].map(label),
    "Check": laatste["id"].str.split("_").str[0] + " · " + laatste["titel"],
    "Dimensie": laatste["dimensie"],
    "Populatie": laatste["populatie"],
    "Bevindingen": laatste["fout"].map(getal),
    "Totaal": laatste["totaal"].map(getal),
    "%": laatste["pct_fout"].map(lambda x: "" if pd.isna(x) else f"{x:.2f}".replace(".", ",")),
    "Δ vorige run": laatste["verschil"].map(lambda x: getal(x, teken=True)),
    "Lijst": laatste["id"].map(lijst_link),
    "Melding": laatste["melding"].fillna(""),
})
display(HTML(
    tabel.to_html(index=False, escape=False, border=0)
    .replace('<table class="dataframe">',
             f'<table style="border-collapse:collapse;font-family:sans-serif;font-size:13px">')
    .replace("<td>", f'<td style="padding:4px 10px;border-bottom:1px solid {RASTER};vertical-align:top;text-align:left;background:transparent">')
    .replace("<th>", f'<th style="padding:4px 10px;text-align:left;color:{INKT_2};border-bottom:1px solid {INKT_2}">')
))

## Lijsten per check

De objecten achter de bevindingen, volgens de laatste run. Klik op een check om de eerste 25 rijen te zien; de volledige lijst staat in de CSV. Monumenten linken naar het monumentenregister.

In [ ]:
import html

def cel(kolom, waarde):
    if pd.isna(waarde) or waarde == "":
        return ""
    w = str(waarde)
    if w.startswith("http"):
        tekst = "register" if kolom == "monumentenregister" else w.rstrip("/").rsplit("/", 1)[-1]
        return f'<a href="{html.escape(w)}" target="_blank">{html.escape(tekst)}</a>'
    return html.escape(w)

blokken = []
for _, r in laatste[laatste["oordeel"] == "bevinding"].iterrows():
    p = lijst_pad(r["id"])
    if not p.exists():
        continue
    df = pd.read_csv(p, dtype=str)
    kop = "".join(f'<th style="padding:3px 8px;text-align:left;color:{INKT_2};border-bottom:1px solid {INKT_2}">{html.escape(k)}</th>' for k in df.columns)
    rijen = "".join(
        "<tr>" + "".join(f'<td style="padding:3px 8px;border-bottom:1px solid {RASTER};text-align:left;background:transparent">{cel(k, v)}</td>' for k, v in rij.items()) + "</tr>"
        for _, rij in df.head(25).iterrows())
    meer = f'<p style="color:{INKT_2}">… en nog {getal(len(df) - 25)} rijen in de CSV.</p>' if len(df) > 25 else ""
    blokken.append(f"""
      <details id="lijst-{r['id']}" style="margin:8px 0;border:1px solid {RASTER};border-radius:8px;padding:8px 12px">
        <summary style="cursor:pointer"><b>{html.escape(r['id'].split('_')[0])} · {html.escape(r['titel'])}</b>
          · {getal(len(df))} rijen · <a href="{p.as_posix()}" download>CSV downloaden</a></summary>
        <div style="overflow-x:auto"><table style="border-collapse:collapse;font-size:12px;margin-top:8px">
          <tr>{kop}</tr>{rijen}</table></div>{meer}
      </details>""")

display(HTML('<div style="font-family:sans-serif;font-size:13px">' + ("".join(blokken) or "Geen lijsten.") + "</div>"
             # Een klik op "bekijk" in de tabel springt naar de lijst en klapt hem open
             + "<script>function openLijst(){var d=document.getElementById(decodeURIComponent(location.hash.slice(1)));if(d&&d.tagName==='DETAILS')d.open=true;}"
               "window.addEventListener('hashchange',openLijst);openLijst();</script>"))

## Verloop per check met bevindingen

Eén grafiek per check, elk met een eigen schaal. Na één run staat er per check alleen één punt; de lijn groeit met elke volgende run.

In [ ]:
ids = laatste.loc[laatste["oordeel"] == "bevinding", "id"].tolist()
if not ids:
    print("Geen checks met bevindingen.")
else:
    kol = 3
    rij = -(-len(ids) // kol)
    import matplotlib.dates as mdates
    from matplotlib.ticker import FuncFormatter, MaxNLocator
    fig, assen = plt.subplots(rij, kol, figsize=(4.2 * kol, 2.6 * rij), squeeze=False)
    for ax, cid in zip(assen.flat, ids):
        d = h[(h["id"] == cid) & (h["status"] == "ok")].sort_values("run")
        ax.plot(d["run"], d["fout"], color=SERIE, linewidth=2, marker="o", markersize=5)
        ax.set_title(d["titel"].iloc[-1], fontsize=9, color=INKT, loc="left", wrap=True)
        ax.annotate(f'{int(d["fout"].iloc[-1]):,}'.replace(",", "."),
                    (d["run"].iloc[-1], d["fout"].iloc[-1]),
                    textcoords="offset points", xytext=(6, 4), fontsize=9, color=INKT)
        ax.set_ylim(bottom=0, top=max(d["fout"].max() * 1.25, 1))
        ax.grid(axis="y", color=RASTER, linewidth=0.8)
        for kant in ("top", "right", "left"):
            ax.spines[kant].set_visible(False)
        ax.spines["bottom"].set_color(INKT_2)
        ax.tick_params(colors=INKT_2, labelsize=8, length=0)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(v):,}".replace(",", ".")))
        if d["run"].nunique() == 1:
            # Eén run: één tik op die datum in plaats van een lege tijdas
            t = d["run"].iloc[0]
            ax.set_xlim(t - pd.Timedelta(days=1), t + pd.Timedelta(days=1))
            ax.set_xticks([t])
        else:
            ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%m-%Y"))
    for ax in list(assen.flat)[len(ids):]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()

## Actualiteit van de graphs

Aantal triples en importdatum volgens de TriplyDB-API. Een lege importdatum betekent dat de graph via een andere route wordt bijgewerkt (zoals instanties-rce, dagelijks).

In [ ]:
g = pd.read_csv("data/graphs_historie.csv")
g["run"] = pd.to_datetime(g["run"])
g_runs = sorted(g["run"].unique())
nu = g[g["run"] == g_runs[-1]].set_index("graph")
if len(g_runs) > 1:
    nu["Δ triples"] = nu["triples"] - g[g["run"] == g_runs[-2]].set_index("graph")["triples"]
else:
    nu["Δ triples"] = float("nan")

HOOFD = {"https://linkeddata.cultureelerfgoed.nl/graph/instanties-rce": "hoofdgraph",
         "https://triplydb.com/koop/owms/graphs/default": "naslag",
         "https://linkeddata.cultureelerfgoed.nl/graph/owms": "naslag"}
nu["Rol"] = nu.index.map(lambda x: HOOFD.get(x, "buiten scope"))
nu["Rol"] = pd.Categorical(nu["Rol"], ["hoofdgraph", "naslag", "buiten scope"], ordered=True)
nu = nu.sort_values(["Rol", "triples"], ascending=[True, False])
overzicht = pd.DataFrame({
    "Rol": nu["Rol"],
    "Graph": nu.index.str.replace(r"^https?://[^/]+/", "", regex=True),
    "Triples": nu["triples"].map(getal),
    "Δ triples": nu["Δ triples"].map(lambda x: getal(x, teken=True)),
    "Geïmporteerd": nu["geimporteerd"].fillna("").str[:10],
})
display(HTML(
    overzicht.to_html(index=False, border=0)
    .replace('<table class="dataframe">', '<table style="border-collapse:collapse;font-family:sans-serif;font-size:13px">')
    .replace("<td>", f'<td style="padding:3px 10px;border-bottom:1px solid {RASTER};text-align:left;background:transparent">')
    .replace("<th>", f'<th style="padding:3px 10px;text-align:left;color:{INKT_2};border-bottom:1px solid {INKT_2}">')
))